# Read in the data

In [ ]:
import pandas as pd
import numpy
import re

data_files = [
    "ap_2010.csv",
    "class_size.csv",
    "demographics.csv",
    "graduation.csv",
    "hs_directory.csv",
    "sat_results.csv"
]
data = {}

for f in data_files:
    key_name = f.replace(".csv", "")
    d = pd.read_csv(f"schools/{f}")
    data[key_name] = d

# Read in the surveys

In [ ]:
all_survey = pd.read_csv("schools/survey_all.txt", delimiter="\t", encoding='windows-1252')
d75_survey = pd.read_csv("schools/survey_d75.txt", delimiter="\t", encoding='windows-1252')
survey = pd.concat([all_survey, d75_survey], axis=0)

survey = survey.copy()
survey["DBN"] = survey["dbn"]

survey_fields = [
    "DBN", 
    "rr_s", 
    "rr_t", 
    "rr_p", 
    "N_s", 
    "N_t", 
    "N_p", 
    "saf_p_11", 
    "com_p_11", 
    "eng_p_11", 
    "aca_p_11", 
    "saf_t_11", 
    "com_t_11", 
    "eng_t_11", 
    "aca_t_11", 
    "saf_s_11", 
    "com_s_11", 
    "eng_s_11", 
    "aca_s_11", 
    "saf_tot_11", 
    "com_tot_11", 
    "eng_tot_11", 
    "aca_tot_11",
]
survey = survey[survey_fields]
data["survey"] = survey

# Add DBN columns

In [ ]:
data["hs_directory"]["DBN"] = data["hs_directory"]["dbn"]

def pad_csd(num):
    string_representation = str(num)
    if len(string_representation) > 1:
        return string_representation
    else:
        return "0" + string_representation
    
data["class_size"]["padded_csd"] = data["class_size"]["CSD"].apply(pad_csd)
data["class_size"]["DBN"] = data["class_size"]["padded_csd"] + data["class_size"]["SCHOOL CODE"]

# Convert columns to numeric

In [ ]:
cols = ['SAT Math Avg. Score', 'SAT Critical Reading Avg. Score', 'SAT Writing Avg. Score']
for c in cols:
    data["sat_results"][c] = pd.to_numeric(data["sat_results"][c], errors="coerce")

data['sat_results']['sat_score'] = data['sat_results'][cols[0]] + data['sat_results'][cols[1]] + data['sat_results'][cols[2]]

def find_lat(loc):
    coords = re.findall("\(.+, .+\)", loc)
    lat = coords[0].split(",")[0].replace("(", "")
    return lat

def find_lon(loc):
    coords = re.findall("\(.+, .+\)", loc)
    lon = coords[0].split(",")[1].replace(")", "").strip()
    return lon

data["hs_directory"]["lat"] = data["hs_directory"]["Location 1"].apply(find_lat)
data["hs_directory"]["lon"] = data["hs_directory"]["Location 1"].apply(find_lon)

data["hs_directory"]["lat"] = pd.to_numeric(data["hs_directory"]["lat"], errors="coerce")
data["hs_directory"]["lon"] = pd.to_numeric(data["hs_directory"]["lon"], errors="coerce")

# Condense datasets

In [ ]:
class_size = data["class_size"]
class_size = class_size[class_size["GRADE "] == "09-12"]
class_size = class_size[class_size["PROGRAM TYPE"] == "GEN ED"]

class_size = class_size.groupby("DBN").agg('mean', numeric_only=True)
class_size.reset_index(inplace=True)
data["class_size"] = class_size

data["demographics"] = data["demographics"][data["demographics"]["schoolyear"] == 20112012]

data["graduation"] = data["graduation"][data["graduation"]["Cohort"] == "2006"]
data["graduation"] = data["graduation"][data["graduation"]["Demographic"] == "Total Cohort"]

# Convert AP scores to numeric

In [ ]:
cols = ['AP Test Takers ', 'Total Exams Taken', 'Number of Exams with scores 3 4 or 5']

for col in cols:
    data["ap_2010"][col] = pd.to_numeric(data["ap_2010"][col], errors="coerce")

# Combine the datasets

In [ ]:
combined = data["sat_results"]

combined = combined.merge(data["ap_2010"], on="DBN", how="left")
combined = combined.merge(data["graduation"], on="DBN", how="left")

to_merge = ["class_size", "demographics", "survey", "hs_directory"]

for m in to_merge:
    combined = combined.merge(data[m], on="DBN", how="inner")

combined = combined.fillna(combined.mean(numeric_only=True))
combined = combined.infer_objects(copy=False).fillna(0)

# Add a school district column for mapping

In [ ]:
def get_first_two_chars(dbn):
    return dbn[0:2]

combined = combined.copy()
combined["school_dist"] = combined["DBN"].apply(get_first_two_chars)

# Find correlations

In [ ]:
correlations = combined.corr(numeric_only=True)
correlations = correlations["sat_score"]
print(correlations)

# Plotting survey correlations

In [ ]:
# Remove DBN since it's a unique identifier, not a useful numerical value for correlation.
survey_fields.remove("DBN")

# Analysis

## Survey Fields vs. SAT Scores

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fields = [f for f in survey_fields if f != 'DBN']
survey_corr = combined.corr(numeric_only=True)['sat_score'][fields]
print(survey_corr.sort_values())

survey_corr.plot.bar(figsize=(10, 5), title='Correlation of survey fields with SAT score')
plt.show()

The strongest positive correlations with `sat_score` come from **`N_s` (0.42) and `N_p` (0.42)**, the number of student and parent survey respondents. These are closely tied to `total_enrollment`, which is itself correlated with SAT scores, so larger schools tend to score higher.

Other observations:

- `rr_s`, the student response rate, correlates positively (0.23). Students who are more likely to fill out surveys may also be more engaged academically.
- How students and teachers perceive **safety** (`saf_s_11` 0.34, `saf_t_11` 0.31, `saf_tot_11` 0.32) correlates with SAT scores. It's hard to teach or learn in an unsafe environment.
- **Academic expectations as seen by students** (`aca_s_11`, 0.34) correlate with SAT scores, while the same metric reported by parents or teachers correlates much more weakly.
- Parent-reported metrics are weak overall (`saf_p_11` 0.12, `com_p_11` -0.12), and teacher response rate (`rr_t`) shows essentially no correlation.

## Safety and SAT Scores

In [ ]:
combined.plot.scatter(x='saf_s_11', y='sat_score', title='Student safety score vs. SAT score')
plt.show()

In [ ]:
boro_safety = combined.groupby('boro')['saf_s_11'].mean().sort_values()
print(boro_safety)

The scatter plot shows a positive but fairly weak relationship between safety score and SAT score. A few schools with very high SAT scores (above 1,800) also have high safety scores, and there are almost no schools with a safety score below 6.5 that have an average SAT above 1,500. Most schools, however, cluster between 1,000 and 1,400 regardless of safety.

Averaged by borough, **Manhattan (6.83) and Queens (6.72)** have the highest student safety scores, while **Brooklyn (6.37)** has the lowest, followed by Staten Island (6.53) and the Bronx (6.61). The differences between boroughs are relatively small, so the safety differences are more pronounced between individual schools than between boroughs.

## Race and SAT Scores

In [ ]:
race_fields = ['white_per', 'asian_per', 'black_per', 'hispanic_per']
race_corr = combined.corr(numeric_only=True)['sat_score'][race_fields]
print(race_corr)

race_corr.plot.bar(title='Correlation of race percentages with SAT score')
plt.show()

In [ ]:
combined.plot.scatter(x='hispanic_per', y='sat_score', title='Hispanic percentage vs. SAT score')
plt.show()

In [ ]:
print(combined[combined['hispanic_per'] > 95][['SCHOOL NAME', 'sat_score', 'ell_percent']])

In [ ]:
print(combined[(combined['hispanic_per'] < 10) & (combined['sat_score'] > 1800)][['SCHOOL NAME', 'sat_score']])

A higher percentage of **white (0.62) or Asian (0.57)** students at a school correlates positively with SAT scores, whereas a higher percentage of **black (-0.28) or Hispanic (-0.40)** students correlates negatively. This may be due to a lack of funding for schools in certain areas, which are more likely to have a higher percentage of black or Hispanic students.

The schools with over 95% Hispanic students all have low SAT scores (887–1,174), and most of them have a very high share of English language learners (`ell_percent` between 72.6% and 94.6% for seven of the eight schools). Many appear to be schools for recent immigrants learning English, which would explain the lower SAT scores — the test is in English.

The schools with less than 10% Hispanic students and an average SAT above 1,800 are **Stuyvesant, Bronx High School of Science, Brooklyn Technical, Queens High School for the Sciences at York College and Staten Island Technical**. These are specialized science and technology schools that admit students through an entrance exam, so their high scores reflect selective admission rather than race itself.

## Gender and SAT Scores

In [ ]:
gender_fields = ['male_per', 'female_per']
gender_corr = combined.corr(numeric_only=True)['sat_score'][gender_fields]
print(gender_corr)

gender_corr.plot.bar(title='Correlation of gender percentages with SAT score')
plt.show()

In [ ]:
combined.plot.scatter(x='female_per', y='sat_score', title='Female percentage vs. SAT score')
plt.show()

In [ ]:
print(combined[(combined['female_per'] > 60) & (combined['sat_score'] > 1700)][['SCHOOL NAME', 'female_per', 'sat_score']])

The correlations are weak in both directions: a higher percentage of females at a school correlates slightly positively with SAT score (0.11), and a higher percentage of males slightly negatively (-0.11). Neither is strong enough to be meaningful on its own.

The scatter plot shows no clear pattern either, although there is a cluster of schools with a high female percentage (60–80%) and high SAT scores. The schools with more than 60% females and an average SAT above 1,700 — **Bard High School Early College, Eleanor Roosevelt, Beacon, Fiorello H. LaGuardia (Music & Art) and Townsend Harris** — are selective schools with strong liberal arts or arts programs and rigorous academic standards.

## AP Exams and SAT Scores

In [ ]:
combined['ap_per'] = combined['AP Test Takers '] / combined['total_enrollment']
print(combined[['ap_per', 'sat_score']].corr())

combined.plot.scatter(x='ap_per', y='sat_score', title='Share of students taking AP exams vs. SAT score')
plt.show()

The correlation between the share of students taking AP exams and SAT score is very weak (0.06). The scatter plot does show that the schools with the highest SAT scores (above 1,800) tend to have a relatively high share of AP test takers (roughly 20–55%), but there are also many schools with a high AP share and average SAT scores. The horizontal line of points around 1,223 comes from schools with missing SAT data that were filled with the column mean, which weakens the measured relationship. Overall, the share of AP test takers alone isn't a good predictor of SAT scores.

## Conclusion

## Summary

- **Survey data:** the number of survey respondents (a proxy for school size), students' perception of safety and academic expectations correlate most with SAT scores; parent responses matter much less.
- **Safety:** safer schools tend to have higher SAT scores, though the relationship is modest. Manhattan and Queens have the highest average safety scores and Brooklyn the lowest.
- **Race:** schools with higher percentages of white and Asian students tend to have higher SAT scores, while schools with more Hispanic and black students tend to score lower. The lowest-scoring schools serve mostly English language learners, and the highest-scoring schools are selective specialized schools.
- **Gender:** there's no meaningful correlation between gender percentages and SAT scores.
- **AP exams:** the share of students taking AP exams is only very weakly related to SAT scores.

Possible next steps: look at class size (`AVERAGE CLASS SIZE`) versus SAT scores, investigate property values in each district as a driver of school quality, and build a score combining SAT results with safety and other factors to find the best schools in each district.